In [ ]:
import numpy as np
import tensorflow as tf
import csv
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from scipy.optimize import minimize

In [ ]:
data = []

with open('original_training_data.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)
    for row in reader:
        float_row = [float(item) for item in row[1:]]
        data.append(float_row)

data = np.array(data)
print(data[0,0:6],data[0,7])


In [ ]:
param_bounds = [
    (0, 1),      # IR
    (50, 500),   # NG
    (50, 500),   # PS
    (0, 1),      # PC
    (0, 1),      # PM
    (3, 15)      # NMP
]

def normalize_params(X, bounds):
    X_norm = np.empty_like(X)
    for i, (min_val, max_val) in enumerate(bounds):
        X_norm[:, i] = (X[:, i] - min_val) / (max_val - min_val)
    return X_norm

def denormalize_params(X_norm, bounds):
    X = np.empty_like(X_norm)
    for i, (min_val, max_val) in enumerate(bounds):
        X[:, i] = X_norm[:, i] * (max_val - min_val) + min_val
    return X

In [ ]:

X = data[:, 0:6]  # Hyperparameters
y = data[:, 7:]  # Results

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
X_normalized = normalize_params(X, param_bounds)
print(X_normalized)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

model = keras.Sequential([
    layers.Input(shape=(1,)),         # Only one input feature: the result
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(6, activation='sigmoid')  # 6 outputs = hyperparameters (normalized 0–1)
])


model.compile(optimizer='adam', loss='mse')

In [ ]:
model.fit(y_scaled, X_normalized, epochs=500, validation_split=0.2, verbose=0)

In [ ]:
import jpype
import jpype.imports

def postprocess_params(params):
    # Ensure it's a clean copy and 1D
    params = params.flatten().copy()

    # Process each parameter individually
    ps = int(round(params[2]))
    ps = max(50, min(ps, 500))
    if ps % 2 != 0:
        ps += 1 if ps < 500 else -1

    nmp = int(round(params[5]))
    nmp = max(3, min(nmp, 15))

    ng = int(round(params[1]))
    ng = max(50, min(ng, 500))

    ir = float(np.clip(params[0], 0, 1))
    pc = float(np.clip(params[3], 0, 1))
    pm = float(np.clip(params[4], 0, 1))

    numeric_params = np.array([ir, ng, ps, pc, pm, nmp])

    # String version for Java input
    java_params = [
        f"{ir:.6f}",  # float
        str(ng),      # int
        str(ps),      # int
        f"{pc:.6f}",  # float
        f"{pm:.6f}",  # float
        str(nmp)      # int
    ]

    return numeric_params, java_params

def run_java_algorithm(params):
    if not jpype.isJVMStarted():
        jpype.startJVM(classpath=["C:/Users/USER/Desktop/my_projects/optimization_with_java/bin"])
    GGA = jpype.JClass("GGA.GGA")  # Just the class name
    java_params = jpype.JArray(jpype.JString)([str(p) for p in params])
    print("→ Running Java GGA with params:", java_params)
    return GGA.main(java_params)

def objective(scaled_result_input):
    # Reshape input (1 feature)
    scaled_result_input = np.array(scaled_result_input).reshape(1, -1)
    
    # Predict normalized hyperparameters
    predicted_params_norm = model.predict(scaled_result_input, verbose=0)

    # Denormalize to real parameter space
    predicted_params = denormalize_params(predicted_params_norm, param_bounds)

    # Optionally postprocess (round integers, etc.)
    predicted_params ,j= postprocess_params(predicted_params)

    # Evaluate GGA algorithm on MKP using these predicted parameters
     # Get numeric params only
    score = run_java_algorithm(j)  # You define this

    return -score  # Negate if you want to maximize


# Start from mean result (scaled)
initial_result = scaler_y.transform(np.mean(y).reshape(1, -1))[0]

# Bounds for input result ∈ scaled space (roughly [-2, 2] for standardized data)
bounds = [(-2, 2)]  # only one input dimension now

# Run optimization
result = minimize(objective, initial_result, bounds=bounds, method='L-BFGS-B')

# Best result input (scaled)
best_result_scaled = result.x.reshape(1, -1)

# Predict best hyperparameters from it
best_params_norm = model.predict(best_result_scaled, verbose=0)
best_params = denormalize_params(best_params_norm, param_bounds)
best_params = postprocess_params(best_params)



print("✅ Best parameters found (original scale):", best_params)
print("🎯 initial resul", initial_result)

In [ ]:
import numpy as np
import subprocess
import csv
from scipy.optimize import minimize


def read_new_training_data(filepath='training_data.csv'):
    new_data = []
    with open(filepath, newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            float_row = [float(item) for item in row[1:]]
            new_data.append(float_row)
    return np.array(new_data)

def retrain_model(model, scaler_X, scaler_y, new_data):
    X_new = new_data[:, 0:6]
    y_new = new_data[:, 7:]
    X_normalized = normalize_params(X_new, param_bounds)
    y_new_scaled = scaler_y.transform(y_new.reshape(-1, 1))
    model.fit(y_new_scaled,X_normalized , epochs=50, validation_split=0.2)

def optimize_params():
    initial_result = scaler_y.transform(np.mean(y).reshape(1, -1))[0]
    bounds = [(-2, 2)] 
    result = minimize(objective, initial_result, bounds=bounds, method='L-BFGS-B')
    return result.x.reshape(1, -1)


# ==== Main Retraining Loop ====
average_result = 0
max_average = -np.inf  # better start with very low
best_possible_params = None
num_iterations = 5  # Number of retrain cycles

for iteration in range(num_iterations):
    print(f"\n--- Iteration {iteration + 1} ---")

    # Step 1: Optimize best parameters based on the current model
    best_params_norm = optimize_params()  # Make sure optimize_params takes all required args!
    best_params_norm = model.predict(best_params_norm, verbose=0)
    best_params = denormalize_params(best_params_norm.reshape(1, -1), param_bounds)

    # Step 2: Postprocess (Java & numeric)
    numeric_params, java_params = postprocess_params(best_params)
    print("→ Optimized & Postprocessed Params:", java_params)

    # Step 3: Run Java algorithm with these parameters
    run_java_algorithm(java_params)

    # Step 4: Read new training data from Java output or other source
    new_training_data = read_new_training_data()
    print("→ New training data shape:", new_training_data.shape)

    # Calculate average result assuming it is in the last column
    average_result = np.mean(new_training_data[:, -1])
    print(f"→ Average Result in Training Data: {average_result:.4f}")

    # Track the best parameters if the current average is better
    if average_result > max_average:
        max_average = average_result
        best_possible_params = numeric_params.copy()  # copy to avoid reference issues
        print(f"→ New Maximum Result Found: {max_average:.4f}")

    # Step 5: Retrain your model with new data
    retrain_model(model, scaler_X, scaler_y, new_training_data)

    # Step 6: After retraining, re-optimize again for refined parameters
    best_params_norm = optimize_params()
    best_params = denormalize_params(best_params_norm.reshape(1, -1), param_bounds)
    numeric_params, java_params = postprocess_params(best_params)

    # Step 7: Predict accuracy with the model for the refined parameters
    normalized_numeric = normalize_params(numeric_params.reshape(1, -1), param_bounds)
    best_params_scaled = scaler_X.transform(normalized_numeric)
    pred_scaled = model.predict(best_params_scaled)
    pred = scaler_y.inverse_transform(pred_scaled)

    print("→ Generated Parameters After Retraining:")
    print(f"   IR  = {numeric_params[0]:.3f}   NG  = {int(numeric_params[1])} PS  = {int(numeric_params[2])} (even)PC  = {numeric_params[3]:.3f} PM  = {numeric_params[4]:.3f} NMP = {int(numeric_params[5])}")
    print(f"   Predicted Accuracy: {pred[0, 0]:.3f}")

print("\nBest possible parameters found during iterations:")
print(f"IR  = {best_possible_params[0]:.3f}   NG  = {int(best_possible_params[1])} PS  = {int(best_possible_params[2])} (even)PC  = {best_possible_params[3]:.3f} PM  = {best_possible_params[4]:.3f} NMP = {int(best_possible_params[5])}")
print(f"Maximum Average Result: {max_average:.4f}")
